# Download model and set up

In [ ]:
from gamadhani.utils.utils import download_models

In [ ]:
hf_model_repo_id = "kmaneeshad/GaMaDHaNi"

pitch_path, qt_path_pitch, audio_path, qt_path_p2a = download_models(hf_model_repo_id, "diffusion")

In [ ]:
print(pitch_path, audio_path)

# Set up data module

In [ ]:
import torch
from torch.utils.data import Dataset
from gamadhani.src.dataset import Task
from typing import Optional, Tuple, Dict
import torchaudio
from scipy.signal import sawtooth
import numpy as np
import torch
import matplotlib.pyplot as plt
from gamadhani.utils.generate_utils import load_pitch_fns, load_audio_fns
from sklearn.preprocessing import QuantileTransformer

In [ ]:
pitch_model, pitch_qt, pitch_task_fn, invert_pitch_fn, primes = load_pitch_fns(
    pitch_path=pitch_path, 
    model_type = 'diffusion', 
    qt_path = qt_path_pitch,
    config_path = 'configs/diffusion_pitch_config.gin',
    device='cuda')  

  

Create a toy dataset of beat, pitch and audio

In [ ]:
pitch_task_fn

In [ ]:
class BeatDataset(Dataset):
    def __init__(
        self,
        pitch_qt: Optional[QuantileTransformer] = None,
        duration: float = 12,
        midi_range: Tuple[int, int] = (38, 81),
        osc_duration: float = 0.5,
    ):
        self.duration = duration
        self.osc_duration = 1
        self.midi_range = midi_range
        self.pitch_qt = pitch_qt
    def midi_to_hz(self, midi_note: int) -> float:
        return 440 * (2 ** ((midi_note - 69) / 12))

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:

        np.random.seed(index)

        # Generate beat signal
        beat_signal = np.zeros(self.duration * 100) # 100 Hz sampling rate
        num_beats = np.random.randint(10, 20)
        beat_locations = np.random.randint(0, self.duration * 100, size=num_beats)
        beat_locations = np.sort(beat_locations)
        beat_signal[beat_locations] = 1
        beat_signal = np.convolve(beat_signal, np.ones(5), mode='same')
        beat_signal = np.clip(beat_signal, 0, 1)

        # Generate pitch signal
        note1, note2 = np.random.choice(range(self.midi_range[0], self.midi_range[1]), size=2, replace=False)
        fs = [self.midi_to_hz(note1), self.midi_to_hz(note2)]
        pitch_signal = np.zeros(self.duration * 100)
        pitch_change_locations = np.concatenate([
            np.array([0]), 
            beat_locations, 
            np.array([pitch_signal.shape[0] - 1])
        ])
        for i in range(1, len(pitch_change_locations)):
            pitch_val = fs[i % 2]
            pitch_signal[pitch_change_locations[i - 1]:pitch_change_locations[i]] = pitch_val + np.random.randn(pitch_change_locations[i] - pitch_change_locations[i - 1]) * 0.01

        normalized_pitch = pitch_task_fn(inputs = {'pitch': {
            'data': pitch_signal,
            }}, time_downsample = 1, qt_transform = self.pitch_qt)['sampled_sequence']

        # Generate audio signal
        pitch_signal_resampled = np.interp(np.linspace(0, 1, 16000*12), np.linspace(0, 1, 1200), pitch_signal)
        win = max(1, int(0.005 * 16000))  # 50ms window to reduce clicks
        kernel = np.ones(win) / win
        f0_smooth = np.convolve(pitch_signal_resampled, kernel, mode="same")
        audio = np.sin(2 * np.pi * np.cumsum(f0_smooth) / 16000)

        return {
            "beat": torch.tensor(beat_signal, dtype=torch.float32),
            "pitch": torch.tensor(pitch_signal, dtype=torch.float32),
            "normalized_pitch": torch.tensor(normalized_pitch, dtype=torch.float32),
            "audio": torch.tensor(audio, dtype=torch.float32),
        }
    
    def __len__(self):
        return 1000


In [ ]:
beat_ds = BeatDataset(pitch_qt = pitch_qt)
beat_data = beat_ds[1]

fig, axs = plt.subplots(3, 1, figsize=(10, 5))
axs[0].plot(beat_data['beat'])
axs[1].plot(beat_data['pitch'])
axs[2].plot(beat_data['normalized_pitch'])
print('test')
ipd.Audio(beat_data['audio'], rate=16000)


# Define model with control layer

In [ ]:
from gamadhani.src.model_diffusion import UNet
from torch import nn

# Finetune the model

In [ ]:
# Robust version: wrap the trained pitch_model directly, add beat conditioning, finetune all params
import copy
import torch
import torch.nn as nn


class BeatConditionedUNet(nn.Module):
    def __init__(self, pretrained_unet: UNet, beat_dim: int = 1, beat_dropout: float = 0.1):
        super().__init__()
        # Keep exact pretrained architecture + weights
        self.unet = copy.deepcopy(pretrained_unet)
        self.beat_projection = nn.Linear(beat_dim, self.unet.initial_projection.out_channels)
        self.beat_dropout = nn.Dropout(beat_dropout)
        self.unet.inp_dim = 1 # hack because this is not set in the model

    @property
    def device(self):
        return next(self.parameters()).device

    def forward(self, x, time, beat, drop=True):
        # INITIAL PROJECTION
        x = self.unet.initial_projection(x)

        # BEAT CONDITIONING
        if beat.ndim == 3:
            beat = beat.transpose(1, 2)  # [B, T, 1]
        elif beat.ndim == 2:
            beat = beat.unsqueeze(-1)    # [B, T, 1]
        else:
            raise ValueError(f"Expected beat shape [B, T] or [B, 1, T], got {beat.shape}")

        beat = self.beat_projection(beat).transpose(1, 2)  # [B, C, T]
        if drop:
            beat = self.beat_dropout(beat)
        x = x + beat

        # TIME CONDITIONING
        time = self.unet.positional_encoding(time)

        def _concat_time(x_, time_):
            time_ = time_.unsqueeze(2).expand(-1, -1, x_.shape[-1])
            return torch.cat([x_, time_], dim=-2)

        skips = []

        # DOWNSAMPLING
        for downsample_layer in self.unet.downsample_layers:
            skips.append(x)
            x = _concat_time(x, time)
            x = downsample_layer(x)
        skips.append(x)

        # BOTTLENECK ATTENTION
        x = x.permute(0, 2, 1)
        x = self.unet.attention_layers(x)
        x = x.permute(0, 2, 1)

        # UPSAMPLING
        for upsample_layer in self.unet.upsample_layers:
            x = _concat_time(x, time)
            x = torch.cat([x, skips.pop(-1)], dim=1)
            x = upsample_layer(x)
        x = torch.cat([x, skips.pop(-1)], dim=1)

        # FINAL PROJECTION
        return self.unet.final_projection(x)

    def loss(self, x, beat):
        padded_x, padding = self.unet.pad_to(x, self.unet.strides_prod)
        padded_beat, _ = self.unet.pad_to(beat, self.unet.strides_prod)

        t = torch.rand((padded_x.shape[0],), device=padded_x.device)
        noise = torch.randn_like(padded_x)
        x_t = t[:, None, None] * padded_x + (1.0 - t[:, None, None]) * noise

        pred = self.forward(x_t, t, padded_beat, drop=self.training)
        pred_unpadded = self.unet.unpad(pred, padding)

        if self.unet.loss_w_padding:
            target = padded_x - noise
            return torch.mean((pred - target) ** 2)

        target = x - self.unet.unpad(noise, padding)
        return torch.mean((pred_unpadded - target) ** 2)

    def sample_cfg(self, batch_size: int, num_steps: int, beat: torch.Tensor, strength = 1):
        # CREATE INITIAL NOISE
        noise = torch.normal(mean=0, std=1, size=(batch_size, self.unet.inp_dim, self.unet.seq_len)).to(self.device)
        padded_noise, padding = self.unet.pad_to(noise, self.unet.strides_prod)
        beat = beat.to(self.device)
        padded_beat, _ = self.unet.pad_to(beat, self.unet.strides_prod)
        
        t_array = torch.ones((batch_size,)).to(self.device)
        with torch.no_grad():
            for t in np.linspace(0, 1, num_steps + 1)[:-1]:
                t_tensor = torch.tensor(t)
                
                unconditioned_logits = self.forward(padded_noise, t_tensor * t_array, padded_beat, drop=False)
                conditioned_logits = self.forward(padded_noise, t_tensor * t_array, padded_beat, drop=False)
                total_logits = strength * conditioned_logits + (1 - strength) * unconditioned_logits
                padded_noise = padded_noise + 1 / num_steps * total_logits
            
            noise = self.unet.unpad(padded_noise, padding)
        return noise


device = 'cuda' if torch.cuda.is_available() else 'cpu'
beat_model = BeatConditionedUNet(pitch_model, beat_dim=1, beat_dropout=0.1).to(device)

# Optional sanity check
print('Trainable params:', sum(p.numel() for p in beat_model.parameters() if p.requires_grad))

In [ ]:
# Finetune ALL weights (base UNet + new beat layers)
from torch.utils.data import DataLoader

train_ds = BeatDataset(pitch_qt = pitch_qt, duration=12)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)

optimizer = torch.optim.AdamW(beat_model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 10

beat_model.train()
losses = []
for epoch in range(num_epochs):
    running_loss = 0.0
    for batch in train_loader:
        # x is target pitch trajectory for diffusion training
        x = batch['normalized_pitch'].to(device).unsqueeze(1)   # [B, 1, T]
        beat = batch['beat'].to(device).unsqueeze(1) # [B, 1, T]

        loss = beat_model.loss(x, beat)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        losses.append(loss.item())

    print(f"epoch {epoch + 1}/{num_epochs} - loss: {running_loss / len(train_loader):.6f}")

# Save finetuned checkpoint
torch.save({'state_dict': beat_model.state_dict()}, 'beat_conditioned_unet_finetuned.ckpt')
print('Saved: beat_conditioned_unet_finetuned.ckpt')

In [ ]:
plt.plot(losses)

In [ ]:
beat_model = BeatConditionedUNet(pitch_model, beat_dim=1, beat_dropout=0.1).to(device)
beat_model.load_state_dict(torch.load('beat_conditioned_unet_finetuned.ckpt')['state_dict'])
beat_model.eval()

In [ ]:
sample = beat_ds[1]
pitch = beat_model.sample_cfg(1, 100, sample['beat'].unsqueeze(0), strength=1)
pitch_unnorm = invert_pitch_fn(**{"f0": pitch.squeeze(0).squeeze(1).cpu().numpy().flatten()}, qt_transform = pitch_qt)

In [ ]:
fig, ax = plt.subplots(3, 1)
ax[0].plot(pitch_unnorm.flatten())
ax[1].plot(sample['pitch'].flatten())
ax[2].plot(sample['beat'].flatten())